# 中学校時間割最適化問題

In [54]:
import pyomo.environ as pyo
from itertools import chain
from collections import defaultdict
from pprint import pprint

In [55]:
TEACHER_COUNTS = {
    "国語": 3, "社会": 2, "数学": 4, "理科": 3, "英語": 4,
    "音楽": 2, "美術": 2, "家庭": 2, "体育": 3, "道徳": 1,
}

N_T = sum(TEACHER_COUNTS.values())

### 集合

In [56]:
D = ["月", "火", "水", "木", "金"]  # 曜日
P = [1, 2, 3, 4, 5, 6]  # 時限
T = ["T" + str(i) for i in range(1, N_T + 1)]  # 教師
T_new = ["T7", "T23"] # 新任教師
S = ["国語", "社会", "数学", "理科", "英語", "音楽", "美術", "家庭", "体育", "道徳"]  # 科目
S_cont = ["音楽", "美術", "家庭"] # 連続時限で行われる科目
S_non_cont = [s for s in S if s not in S_cont] # 連続時限で行われない科目
G = [1, 2, 3]  # 学年
C = [f"{g}年{i}組" for g in G for i in ["A", "B", "C", "D"]]  # クラス

Q_2 = [(1, 2), (2, 3), (3, 4), (5, 6)]  # 同一曜日内の連続2時限のペア（昼休みをまたぐ4限–5限を除外）
Q_4 = [(1, 2, 3, 4), (2, 3, 4, 5), (3, 4, 5, 6)]  # 同一曜日内の連続4時限の組（昼休みをまたぐものも含む）

### パラメータ

In [57]:
r = {
    1: {"国語": 4, "社会": 3, "数学": 5, "理科": 3, "英語": 5, "音楽": 2, "美術": 2, "家庭": 2, "体育": 3, "道徳": 1},
    2: {"国語": 4, "社会": 3, "数学": 5, "理科": 3, "英語": 5, "音楽": 2, "美術": 2, "家庭": 2, "体育": 3, "道徳": 1},
    3: {"国語": 4, "社会": 4, "数学": 5, "理科": 4, "英語": 5, "音楽": 2, "美術": 2, "家庭": 0, "体育": 3, "道徳": 1},
} # 学年gにおける科目sの週あたり時限数

# \sum_{s \in S} r_{g, s} = 30 をチェック
for g in G:
    total = sum(r[g][s] for s in S)
    print(f"学年 {g} の合計: {total}{' NG' if total != 30 else ''}")

学年 1 の合計: 30
学年 2 の合計: 30
学年 3 の合計: 30


In [58]:
g = {c: int(c[0]) for c in C}  # クラスcの所属学年

In [59]:
sigma = {
    f"T{i}": subject
    for i, subject in enumerate(
        chain.from_iterable([s] * n for s, n in TEACHER_COUNTS.items()), start=1
    )
}

In [60]:
teachers_by_subject = defaultdict(list)
for t, s in sigma.items():
    teachers_by_subject[s].append(t)

# 各学年・クラスの (担任科目, 副担任科目) ペア
grade_subject_pairs = {
    1: [("国語", "社会"), ("体育", "理科"), ("英語", "数学"), ("音楽", "家庭")],
    2: [("理科", "英語"), ("数学", "英語"), ("社会", "体育"), ("音楽", "国語")],
    3: [("国語", "美術"), ("数学", "家庭"), ("理科", "体育"), ("数学", "英語")],
}

classes_by_grade = {grade: [c for c in C if g[c] == grade] for grade in G}
teacher_pool = defaultdict(int)  # 科目ごとの割り当てインデックス

h1, h2 = {}, {}
for grade in G:
    for cls, (s1, s2) in zip(classes_by_grade[grade], grade_subject_pairs[grade]):
        h1[cls] = teachers_by_subject[s1][teacher_pool[s1]]
        h2[cls] = teachers_by_subject[s2][teacher_pool[s2]]
        teacher_pool[s1] += 1
        teacher_pool[s2] += 1

In [61]:
assert all(sigma[h1[c]] != sigma[h2[c]] for c in C), "仮定2違反"
assert len(set(list(h1.values()) + list(h2.values()))) == len(C) * 2, "教師重複"

In [62]:
print("担任:")
pprint(h1)
print("副担任:")
pprint(h2)

担任:
{'1年A組': 'T1',
 '1年B組': 'T23',
 '1年C組': 'T13',
 '1年D組': 'T17',
 '2年A組': 'T11',
 '2年B組': 'T7',
 '2年C組': 'T5',
 '2年D組': 'T18',
 '3年A組': 'T3',
 '3年B組': 'T8',
 '3年C組': 'T12',
 '3年D組': 'T9'}
副担任:
{'1年A組': 'T4',
 '1年B組': 'T10',
 '1年C組': 'T6',
 '1年D組': 'T21',
 '2年A組': 'T14',
 '2年B組': 'T15',
 '2年C組': 'T24',
 '2年D組': 'T2',
 '3年A組': 'T19',
 '3年B組': 'T22',
 '3年C組': 'T25',
 '3年D組': 'T16'}


In [63]:
# 新任教師が担任に入っていることを確認
for t in T_new:
    assert t in h1.values(), f"{t} が担任に割り当てられていません"

### モデルの作成

In [64]:
model = pyo.ConcreteModel()

model.N_S = pyo.RangeSet(1, len(S))
model.N_D = pyo.RangeSet(1, len(D))
model.N_P = pyo.RangeSet(1, len(P))
model.N_C = pyo.RangeSet(1, len(C))
model.N_T = pyo.RangeSet(1, N_T)
model.N_G = pyo.RangeSet(1, len(G))

model.x = pyo.Var(model.N_S, model.N_D, model.N_P, model.N_C, domain=pyo.Binary)
model.y = pyo.Var(model.N_T, model.N_C, domain=pyo.Binary)
model.z = pyo.Var(model.N_T, model.N_D, model.N_P, model.N_C, domain=pyo.Binary)
model.y_tilde = pyo.Var(model.N_T, model.N_G, domain=pyo.Binary)
model.w = pyo.Var(model.N_S, model.N_D, model.N_P, model.N_C, domain=pyo.Binary)

### 制約

In [65]:
# インデックス変換テーブル（RangeSet整数 ↔ パラメータ辞書キー）
S_to_idx = {s: i+1 for i, s in enumerate(S)}
T_by_subject_idx = {s: [T.index(t)+1 for t in ts] for s, ts in teachers_by_subject.items()}
h1_t_idx = {i+1: T.index(h1[c])+1 for i, c in enumerate(C)}
h2_t_idx = {i+1: T.index(h2[c])+1 for i, c in enumerate(C)}


def constraint_num_lectures_per_period_per_class(model, d, p, c):
    return sum(model.x[s, d, p, c] for s in model.N_S) <= 1


def constraint_teachers_per_class_per_subject(model, c, s):
    s_name = S[s-1]
    sum_y = sum(model.y[t, c] for t in T_by_subject_idx[s_name])
    return sum_y == 1 if r[g[C[c-1]]][s_name] > 0 else sum_y == 0


def constraint_necessary_periods_for_each_subject(model, s, c):
    return sum(model.x[s, d, p, c] for d in model.N_D for p in model.N_P) == r[g[C[c-1]]][S[s-1]]


def constraint_assignment_homeroom_teachers(model, c):
    c_name = C[c-1]
    t = h1_t_idx[c]
    return model.y[t, c] == 1 if r[g[c_name]][sigma[h1[c_name]]] > 0 else model.y[t, c] == 0


def constraint_assignment_sub_homeroom_teachers(model, c):
    c_name = C[c-1]
    t = h2_t_idx[c]
    return model.y[t, c] == 1 if r[g[c_name]][sigma[h2[c_name]]] > 0 else model.y[t, c] == 0


def constraint_upper_bound_on_num_lectures_per_subject_per_day(model, s, d, c):
    return sum(model.x[s, d, p, c] for p in model.N_P) <= 1 if S[s-1] in S_non_cont else pyo.Constraint.Skip


def constraint_continuous_periods_for_continuous_subjects_1(model, s, c):
    if S[s-1] not in S_cont or r[g[C[c-1]]][S[s-1]] == 0:
        return pyo.Constraint.Skip
    p_start = [q[0] for q in Q_2]
    return sum(model.w[s, d, p, c] for d in model.N_D for p in p_start) == 1


def constraint_continuous_periods_for_continuous_subjects_2(model, s, d, p, c):
    if S[s-1] not in S_cont or (p, p+1) not in Q_2:
        return pyo.Constraint.Skip
    return model.x[s, d, p, c] >= model.w[s, d, p, c]


def constraint_continuous_periods_for_continuous_subjects_3(model, s, d, p, c):
    if S[s-1] not in S_cont or (p, p+1) not in Q_2:
        return pyo.Constraint.Skip
    return model.x[s, d, p+1, c] >= model.w[s, d, p, c]


def constraint_continuous_periods_for_continuous_subjects_4(model, s, d, p, c):
    if S[s-1] not in S_cont or r[g[C[c-1]]][S[s-1]] == 0:
        return pyo.Constraint.Skip
    # p を含む有効ブロック（Q_2）の開始時限一覧
    valid_starts = [q[0] for q in Q_2 if p in q]
    return model.x[s, d, p, c] <= sum(model.w[s, d, ps, c] for ps in valid_starts)


def constraint_relationship_between_x_y_z_1(model, t, d, p, c):
    s_idx = S_to_idx[sigma[f"T{t}"]]
    return model.z[t, d, p, c] >= model.x[s_idx, d, p, c] + model.y[t, c] - 1


def constraint_relationship_between_x_y_z_2(model, t, d, p, c):
    s_idx = S_to_idx[sigma[f"T{t}"]]
    return model.z[t, d, p, c] <= model.x[s_idx, d, p, c]


def constraint_relationship_between_x_y_z_3(model, t, d, p, c):
    return model.z[t, d, p, c] <= model.y[t, c]


def constraint_num_assignments_per_teacher_per_period(model, t, d, p):
    return sum(model.z[t, d, p, c] for c in model.N_C) <= 1


def constraint_lower_bound_on_num_lectures_per_teacher_per_day(model, t, d):
    return sum(model.z[t, d, p, c] for p in model.N_P for c in model.N_C) >= 1


def constraint_prohibit_teacher_four_continuous_periods(model, t, d, p):
    if (p, p+1, p+2, p+3) not in Q_4:
        return pyo.Constraint.Skip
    return sum(model.z[t, d, pp, c] for pp in range(p, p+4) for c in model.N_C) <= 3


def constraint_relationship_between_y_and_y_tilde(model, t, c):
    return model.y_tilde[t, g[C[c-1]]] >= model.y[t, c]


def constraint_limit_on_num_grades_per_new_teacher(model, t):
    if f"T{t}" not in T_new:
        return pyo.Constraint.Skip
    return sum(model.y_tilde[t, gr] for gr in model.N_G) == 1

### 目的関数

In [66]:
def objective_function(model):
    return sum(model.y_tilde[t, gr] for t in model.N_T for gr in model.N_G)

### モデルに制約と目的関数を追加

In [67]:
# 再実行時の重複登録を防ぐ
for _name in ['con1', 'con2', 'con3', 'con4', 'con5', 'con6', 'con7a', 'con7b', 'con7c', 'con8a', 'con8b', 'con8c', 'con9', 'con10', 'con11', 'con12', 'con13']:
    if model.component(_name):
        model.del_component(_name)

# 制約をモデルに追加
model.con1  = pyo.Constraint(model.N_D, model.N_P, model.N_C,             rule=constraint_num_lectures_per_period_per_class)
model.con2  = pyo.Constraint(model.N_C, model.N_S,                        rule=constraint_teachers_per_class_per_subject)
model.con3  = pyo.Constraint(model.N_S, model.N_C,                        rule=constraint_necessary_periods_for_each_subject)
model.con4  = pyo.Constraint(model.N_C,                                   rule=constraint_assignment_homeroom_teachers)
model.con5  = pyo.Constraint(model.N_C,                                   rule=constraint_assignment_sub_homeroom_teachers)
model.con6  = pyo.Constraint(model.N_S, model.N_D, model.N_C,             rule=constraint_upper_bound_on_num_lectures_per_subject_per_day)
# model.con7a = pyo.Constraint(model.N_S, model.N_C,                        rule=constraint_continuous_periods_for_continuous_subjects_1)
# model.con7b = pyo.Constraint(model.N_S, model.N_D, model.N_P, model.N_C, rule=constraint_continuous_periods_for_continuous_subjects_2)
# model.con7c = pyo.Constraint(model.N_S, model.N_D, model.N_P, model.N_C, rule=constraint_continuous_periods_for_continuous_subjects_3)
# model.con7d = pyo.Constraint(model.N_S, model.N_D, model.N_P, model.N_C, rule=constraint_continuous_periods_for_continuous_subjects_4)
model.con8a = pyo.Constraint(model.N_T, model.N_D, model.N_P, model.N_C, rule=constraint_relationship_between_x_y_z_1)
model.con8b = pyo.Constraint(model.N_T, model.N_D, model.N_P, model.N_C, rule=constraint_relationship_between_x_y_z_2)
model.con8c = pyo.Constraint(model.N_T, model.N_D, model.N_P, model.N_C, rule=constraint_relationship_between_x_y_z_3)
model.con9  = pyo.Constraint(model.N_T, model.N_D, model.N_P,             rule=constraint_num_assignments_per_teacher_per_period)
model.con10 = pyo.Constraint(model.N_T, model.N_D,                        rule=constraint_lower_bound_on_num_lectures_per_teacher_per_day)
model.con11 = pyo.Constraint(model.N_T, model.N_D, model.N_P,             rule=constraint_prohibit_teacher_four_continuous_periods)
# model.con12 = pyo.Constraint(model.N_T, model.N_C,                        rule=constraint_relationship_between_y_and_y_tilde)
# model.con13 = pyo.Constraint(model.N_T,                                   rule=constraint_limit_on_num_grades_per_new_teacher)

In [68]:
# 再実行時の重複登録を防ぐ
for _name in list(model.component_map(pyo.Objective).keys()):
    model.del_component(_name)

# 目的関数をモデルに追加
# model.obj = pyo.Objective(rule=objective_function, sense=pyo.minimize)

### 求解

In [69]:
opt = pyo.SolverFactory('highs')

In [70]:
opt.solve(model, tee=True)

{'Problem': [{'Lower bound': -inf, 'Upper bound': inf, 'Number of objectives': 0, 'Number of constraints': nan, 'Number of variables': nan, 'Sense': 'unknown'}], 'Solver': [{'Status': 'ok', 'Termination condition': 'optimal', 'Termination message': 'TerminationCondition.convergenceCriteriaSatisfied'}], 'Solution': [OrderedDict({'number of solutions': 0, 'number of solutions displayed': 0})]}

In [71]:
def format_timetable(model):
    md = []
    for ci, c in enumerate(C):
        c_idx = ci + 1
        md.append(f"## {c}\n")
        md.append("| 時限 | " + " | ".join(D) + " |")
        md.append("|:---:|" + ":---:|" * len(D))
        for p in P:
            row = [str(p)]
            for di in range(len(D)):
                d_idx = di + 1
                cell = ""
                for si, s_name in enumerate(S):
                    if (pyo.value(model.x[si+1, d_idx, p, c_idx], exception=False) or 0) > 0.5:
                        teacher = next(
                            (T[ti] for ti in range(len(T))
                             if (pyo.value(model.z[ti+1, d_idx, p, c_idx], exception=False) or 0) > 0.5),
                            ""
                        )
                        cell = f"{s_name} ({teacher})"
                        break
                row.append(cell)
            md.append("| " + " | ".join(row) + " |")
        md.append("")
    return "\n".join(md)

output_path = "timetable.md"
with open(output_path, "w") as f:
    f.write(format_timetable(model))
print(f"書き出し完了: {output_path}")

書き出し完了: timetable.md


In [72]:
import pandas as pd
from IPython.display import display

def v(expr):
    val = pyo.value(expr, exception=False)
    return val if val is not None else 0.0

# 解を読み出す: timetable[c][(d,p)] = (科目 or None, 教師 or None)
timetable = {c: {} for c in C}
for ci, c in enumerate(C):
    for di, d in enumerate(D):
        for p in P:
            subj = next((S[si] for si in range(len(S))
                         if v(model.x[si+1, di+1, p, ci+1]) > 0.5), None)
            tchr = next((T[ti] for ti in range(len(T))
                         if v(model.z[ti+1, di+1, p, ci+1]) > 0.5), None) if subj else None
            timetable[c][(d, p)] = (subj, tchr)

checks = []

# 1. 授業時数（週あたり）
issues = []
for c in C:
    for s in S:
        count = sum(1 for d in D for p in P if timetable[c][(d, p)][0] == s)
        req = r[g[c]][s]
        if count != req:
            issues.append(f"{c} {s}: {count}時限（要 {req}）")
checks.append(("科目ごと週あたり授業時数", issues))

# 2. 各クラスの1時限に高々1科目
issues = []
for ci, c in enumerate(C):
    for di, d in enumerate(D):
        for p in P:
            cnt = sum(1 for si in range(len(S)) if v(model.x[si+1, di+1, p, ci+1]) > 0.5)
            if cnt > 1:
                issues.append(f"{c}: {d}曜{p}限に{cnt}科目")
checks.append(("各クラスの1時限に高々1科目", issues))

# 3. 教師の重複なし（同時間帯に複数クラス担当）
issues = []
seen = {}
for c in C:
    for d in D:
        for p in P:
            _, tchr = timetable[c][(d, p)]
            if tchr:
                key = (tchr, d, p)
                if key in seen:
                    issues.append(f"{tchr}: {d}曜{p}限に {seen[key]} と {c}")
                seen.setdefault(key, c)
checks.append(("教師は1つの時限に高々1クラスを担当", issues))

# 4. 連続時限科目（音楽・美術・家庭）の連続配置
issues = []
for c in C:
    for s in S_cont:
        slots = [(d, p) for d in D for p in P if timetable[c][(d, p)][0] == s]
        if len(slots) != 2:
            issues.append(f"{c} {s}: {len(slots)}時限")
        else:
            (d1, p1), (d2, p2) = slots
            if d1 != d2 or (p1, p2) not in Q_2:
                issues.append(f"{c} {s}: {d1}曜{p1}限 + {d2}曜{p2}限（連続でない）")
checks.append(("連続時限科目の連続配置", issues))

# 5. 担任・副担任が自クラスを担当
issues = []
for c in C:
    for role, t in [("担任", h1[c]), ("副担任", h2[c])]:
        s = sigma[t]
        if r[g[c]][s] > 0:
            if not any(timetable[c][(d, p)][1] == t for d in D for p in P):
                issues.append(f"{c}: {role} {t}（{s}）が自クラスを担当していない")
checks.append(("担任・副担任が自クラスを担当", issues))

# 6. 新任教師は1学年のみ担当
issues = []
for t in T_new:
    grades = {g[c] for c in C if any(timetable[c][(d, p)][1] == t for d in D for p in P)}
    if len(grades) > 1:
        issues.append(f"{t}（{sigma[t]}）: {sorted(grades)}学年を担当")
checks.append(("新任教師は1学年のみ担当", issues))

# 7. 非連続科目は1日1時限以下
issues = []
for c in C:
    for s in S_non_cont:
        for d in D:
            cnt = sum(1 for p in P if timetable[c][(d, p)][0] == s)
            if cnt > 1:
                issues.append(f"{c}: {d}曜に{s}が{cnt}時限")
checks.append(("非連続科目は1日1時限以下", issues))

# 8. 教師は毎日1時限以上授業
issues = []
for t in T:
    for d in D:
        if not any(timetable[c][(d, p)][1] == t for c in C for p in P):
            issues.append(f"{t}（{sigma[t]}）: {d}曜に授業なし")
checks.append(("教師は毎日1時限以上授業", issues))

# 9. 教師の4時限連続授業の禁止
issues = []
for t in T:
    for d in D:
        for q in Q_4:
            cnt = sum(1 for p in q for c in C if timetable[c][(d, p)][1] == t)
            if cnt > 3:
                issues.append(f"{t}（{sigma[t]}）: {d}曜{q[0]}〜{q[-1]}限すべて担当")
checks.append(("教師の4時限連続授業の禁止", issues))

# サマリ表示
rows = [{"要件": title, "状態": "OK" if not iss else f"NG ({len(iss)} 件)"}
        for title, iss in checks]
df = pd.DataFrame(rows)
display(df.style.map(
    lambda val: "color: green" if val == "OK" else ("color: red; font-weight: bold" if "NG" in str(val) else ""),
    subset=["状態"]
).hide(axis="index"))

# 違反詳細
# for title, issues in checks:
#     if issues:
#         print(f"\n【{title}】")
#         for issue in issues[:5]:
#             print(f"  • {issue}")
#         if len(issues) > 5:
#             print(f"  ... 他 {len(issues) - 5} 件")

要件,状態
科目ごと週あたり授業時数,OK
各クラスの1時限に高々1科目,OK
教師は1つの時限に高々1クラスを担当,OK
連続時限科目の連続配置,NG (35 件)
担任・副担任が自クラスを担当,OK
新任教師は1学年のみ担当,NG (2 件)
非連続科目は1日1時限以下,OK
教師は毎日1時限以上授業,OK
教師の4時限連続授業の禁止,OK
